# 🎵 Music to MIDI — Google Colab (Free GPU)

在 Colab 免费 T4 GPU 上运行 Music to MIDI，通过 Gradio 公开链接访问。

当前 Colab 版提供六种处理模式：
- **完整混音多乐器转写（SMART）**：直接用可选择的 YourMT3+ 官方模型模式对整首音频生成一个 GM 多乐器 MIDI。
- **人声/伴奏分离后分别转写（VOCAL_SPLIT）**：先分离人声与伴奏，再分别转写并可额外合并 MIDI。
- **六声部分离 + 分别转写**：先分离 bass / drums / guitar / piano / vocals / other 六个 stem，再分别生成 stem MIDI 和合并 MIDI。
- **钢琴专用转写 (Transkun)**：使用 Transkun 钢琴专用模型直接转写纯钢琴音频。
- **钢琴专用转写 (Aria-AMT)**：使用 Aria-AMT 钢琴专用模型直接转写纯钢琴音频。

Colab 版仅在 SMART 完整混音模式显示 YourMT3+ 官方模型模式；VOCAL_SPLIT 与六声部分离固定使用工程默认 YourMT3+ noPS 后端完成 MIDI 扩展。钢琴专用入口使用 Transkun、Aria-AMT 或 ByteDance Pedal。桌面版如需 MIROS 后端请在本地运行。

**输出说明**：SMART 输出一个完整混音 MIDI；VOCAL_SPLIT 输出 `_accompaniment.mid` 与 `_vocal.mid`，勾选合并后额外输出 `_vocal_accompaniment_merged.mid`；六声部模式输出各 stem MIDI、分离音频和一个合并 MIDI；钢琴专用模式输出一个钢琴 MIDI。

**使用方法**：
1. 点击上方菜单 **Runtime → Change runtime type → T4 GPU**
2. 依次运行下面的单元格
3. 最后一个单元格会输出一个公开链接，打开即可使用

In [ ]:
#@title 1️⃣ 检查 GPU 可用性（详细日志）
import subprocess
import sys
from datetime import datetime

import torch


def log(message=""):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)


log("Starting runtime diagnostics")
log(f"Python version: {sys.version.split()[0]}")
log(f"PyTorch version: {torch.__version__}")
log(f"Torch CUDA build: {torch.version.cuda}")
log(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_index = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device_index)
    total_vram_gb = props.total_memory / 1024**3
    log(f"Detected GPU: {props.name}")
    log(f"Total VRAM: {total_vram_gb:.2f} GB")
    log(f"SM count: {props.multi_processor_count}")
    try:
        reserved = torch.cuda.memory_reserved(device_index) / 1024**3
        allocated = torch.cuda.memory_allocated(device_index) / 1024**3
        log(f"Allocated VRAM: {allocated:.3f} GB")
        log(f"Reserved VRAM: {reserved:.3f} GB")
    except Exception as exc:
        log(f"Failed to read VRAM stats: {exc}")

    log("nvidia-smi summary:")
    try:
        smi = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,memory.used,driver_version",
                "--format=csv,noheader",
            ],
            text=True,
        ).strip()
        print(smi, flush=True)
    except Exception as exc:
        log(f"Unable to run nvidia-smi: {exc}")

    log("GPU runtime check passed")
else:
    log("No GPU detected. In Colab use Runtime -> Change runtime type -> T4 GPU")
    raise RuntimeError("GPU runtime is required")


In [ ]:
#@title 2️⃣ 克隆仓库并安装依赖（详细日志）
import importlib
import os
import shlex
import subprocess
import time
from datetime import datetime
from pathlib import Path


def log(message=""):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)


def run_cmd(command, cwd=None):
    log(f"CMD: {command}")
    start = time.time()
    proc = subprocess.Popen(
        command,
        cwd=cwd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    line_count = 0
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
        line_count += 1
    return_code = proc.wait()
    elapsed = time.time() - start
    log(f"command finished: exit={return_code}, lines={line_count}, elapsed={elapsed:.1f}s")
    if return_code != 0:
        raise RuntimeError(f"command failed: {command}")


repo_dir = Path("/content/music-to-midi")
repo_url = "https://github.com/mason369/music-to-midi.git"

log("准备项目工作区")
if not repo_dir.exists():
    run_cmd(f"git clone {repo_url} {repo_dir}")
else:
    log("仓库已存在，显示状态并同步远程")
    run_cmd("git -C /content/music-to-midi remote -v")
    run_cmd("git -C /content/music-to-midi rev-parse --short HEAD")
    run_cmd("git -C /content/music-to-midi fetch --all --prune")
    run_cmd("git -C /content/music-to-midi status -sb")

os.chdir(repo_dir)
log(f"工作目录: {os.getcwd()}")

log("升级 pip/setuptools/wheel")
run_cmd("python -m pip install --upgrade pip setuptools wheel")

# ── 记录 Colab 预装的 torch 版本（绝不重装） ──
log("检测 Colab 预装 torch 版本")
import torch
log(f"torch=={torch.__version__}")
log(f"CUDA available: {torch.cuda.is_available()}, CUDA version: {torch.version.cuda}")

log("安装 Python 依赖")
# 保留 Colab 预装 torch；只安装当前 Web 版需要的依赖
packages = [
    "lightning",
    "einops",
    "transformers==4.45.1",
    "mir-eval",
    "deprecated",
    "smart-open",
    "nest-asyncio",
    "scipy",
    "scikit-learn",
    "torchmetrics",
    "wandb",
    "pretty-midi>=0.2.10",
    "soxr>=0.3.7",
    "huggingface_hub",
    "mido",
    "librosa",
    "soundfile",
    "transkun>=2.0.1",
    "aria-amt @ git+https://github.com/EleutherAI/aria-amt.git",
    "piano-transcription-inference>=0.0.6,<0.1",
    "torchlibrosa>=0.1.0,<0.2",
    "matplotlib>=3.7.0,<4",
    "gradio==4.44.1",
    "fastapi==0.115.2",
    "starlette==0.40.0",
    "tqdm",
    "pyyaml",
    "psutil",
    "numba",
]
quoted_packages = " ".join(shlex.quote(pkg) for pkg in packages)
run_cmd("python -m pip install " + quoted_packages)

log("安装 audio-separator 运行依赖（固定兼容 NumPy 1.26）")
audio_separator_runtime_packages = [
    "numpy==1.26.4",
    "beartype==0.18.5",
    "diffq-fixed==0.2.4",
    "julius==0.2.7",
    "ml_collections==1.1.0",
    "onnx-weekly==1.21.0.dev20260302",
    "onnx2torch-py313==1.6.0",
    "pydub==0.25.1",
    "requests>=2.32.5,<3",
    "chardet>=5,<6",
    "onnxruntime==1.23.2",
    "resampy==0.4.3",
    "rotary-embedding-torch==0.6.5",
    "samplerate==0.1.0",
    "h5py>=3.10,<4",
    "mirdata>=0.3.8,<1",
    "six==1.17.0",
]
quoted_audio_separator_runtime_packages = " ".join(shlex.quote(pkg) for pkg in audio_separator_runtime_packages)
run_cmd("python -m pip install " + quoted_audio_separator_runtime_packages)
run_cmd("python -m pip install " + shlex.quote("audio-separator==0.44.1") + " --no-deps")

log("使用 apt 安装 FFmpeg")
run_cmd("apt-get update")
run_cmd("apt-get install -y ffmpeg")

log("关键包版本")
for module_name in ["torch", "torchaudio", "torchvision", "gradio", "huggingface_hub", "lightning", "librosa"]:
    try:
        module = importlib.import_module(module_name)
        version = getattr(module, "__version__", "unknown")
        log(f"{module_name}=={version}")
    except Exception as exc:
        log(f"无法读取 {module_name} 版本: {exc}")

log("依赖安装完成")

In [ ]:
#@title 3️⃣ 下载 YourMT3+ 源码和模型权重（详细日志）
import os
import sys
import time
from datetime import datetime
from pathlib import Path

from huggingface_hub import list_repo_files, snapshot_download

os.chdir("/content/music-to-midi")


def log(message=""):
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {message}", flush=True)


def format_size(num_bytes):
    size = float(num_bytes)
    for unit in ("B", "KB", "MB", "GB"):
        if size < 1024 or unit == "GB":
            if unit == "B":
                return f"{int(size)} {unit}"
            return f"{size:.1f} {unit}"
        size /= 1024
    return f"{size:.1f} GB"


def collect_local_file_info(base_dir, relative_paths):
    file_info = {}
    base_path = Path(base_dir)
    for relative_path in relative_paths:
        local_path = base_path / relative_path
        if local_path.exists():
            file_info[relative_path] = local_path.stat().st_size
    return file_info


repo_id = "mimbres/YourMT3"
yourmt3_dir = "/content/music-to-midi/space/YourMT3"
amt_src = os.path.join(yourmt3_dir, "amt", "src")

log("准备 YourMT3 源码同步（详细模式）")
log(f"目标目录: {yourmt3_dir}")
log(
    "HF_TOKEN 状态: 已设置"
    if os.environ.get("HF_TOKEN")
    else "HF_TOKEN 状态: 未设置（公开仓库可匿名下载）"
)

source_start = time.time()
repo_source_files = sorted(
    file_path
    for file_path in list_repo_files(repo_id, repo_type="space")
    if file_path.startswith("amt/src/")
)

if not repo_source_files:
    raise RuntimeError("远程仓库中未找到 amt/src 文件，请重试。")

log(f"远程源文件数量: {len(repo_source_files)}")
source_before = collect_local_file_info(yourmt3_dir, repo_source_files)
log(f"同步前本地缓存文件数: {len(source_before)}")
log("源文件计划（同步前）：")
for index, file_path in enumerate(repo_source_files, 1):
    if file_path in source_before:
        log(f"  [{index:03d}/{len(repo_source_files)}] CACHE {file_path} ({format_size(source_before[file_path])})")
    else:
        log(f"  [{index:03d}/{len(repo_source_files)}] FETCH {file_path}")

snapshot_download(
    repo_id,
    repo_type="space",
    local_dir=yourmt3_dir,
    allow_patterns=["amt/src/**"],
    ignore_patterns=["*.ckpt", "*.bin", "*.safetensors", "amt/logs/**"],
)

source_after = collect_local_file_info(yourmt3_dir, repo_source_files)
new_files = [file_path for file_path in repo_source_files if file_path not in source_before and file_path in source_after]
cached_files = [file_path for file_path in repo_source_files if file_path in source_before and file_path in source_after]

log("源文件结果（同步后）：")
for index, file_path in enumerate(repo_source_files, 1):
    if file_path in source_after:
        status = "NEW" if file_path in new_files else "CACHE"
        log(f"  [{index:03d}/{len(repo_source_files)}] {status:5} {file_path} ({format_size(source_after[file_path])})")
    else:
        log(f"  [{index:03d}/{len(repo_source_files)}] MISSING {file_path}")

source_total_size = sum(source_after.values())
source_elapsed = time.time() - source_start
log(
    f"YourMT3 源码就绪: 新增={len(new_files)}, 缓存={len(cached_files)}, "
    f"总计={format_size(source_total_size)}, 耗时={source_elapsed:.1f}秒"
)

root_link = "/content/music-to-midi/YourMT3"
if not os.path.exists(root_link):
    os.symlink(yourmt3_dir, root_link)
    log("符号链接已创建: YourMT3 -> space/YourMT3")
else:
    log("符号链接已存在: YourMT3 -> space/YourMT3")

if amt_src not in sys.path:
    sys.path.insert(0, amt_src)
    log("已将 amt/src 添加到 sys.path")

log("开始检查/下载 YourMT3+ 官方模型模式权重（首次运行会下载全部可选模式）")
sys.path.insert(0, "/content/music-to-midi")
from download_sota_models import download_official_yourmt3_models

model_cache_root = Path(os.path.expanduser("~/.cache/music_ai_models/yourmt3_all"))
existing_ckpts = sorted(model_cache_root.rglob("*.ckpt")) if model_cache_root.exists() else []
if existing_ckpts:
    log("现有检查点缓存：")
    for idx, ckpt in enumerate(existing_ckpts, 1):
        log(f"  [{idx:02d}/{len(existing_ckpts)}] {ckpt} ({format_size(ckpt.stat().st_size)})")
else:
    log("本地未找到检查点缓存；预计首次下载")

downloaded_yourmt3_models = download_official_yourmt3_models()
log(f"YourMT3+ 官方模型模式准备完成: {len(downloaded_yourmt3_models)} 个")

final_ckpts = sorted(model_cache_root.rglob("*.ckpt")) if model_cache_root.exists() else []
if final_ckpts:
    log("下载后的检查点文件：")
    for idx, ckpt in enumerate(final_ckpts, 1):
        log(f"  [{idx:02d}/{len(final_ckpts)}] {ckpt} ({format_size(ckpt.stat().st_size)})")
else:
    log("警告：下载后未检测到 .ckpt 文件，请检查上方日志。")

log("检查六声部分离、人声分离、Aria-AMT 与 ByteDance Piano 资源（按需下载）")
from download_multistem_model import download_multistem_model
from download_vocal_model import download_vocal_model
from download_vocal_harmony_model import download_chorus_model
from download_aria_amt_model import download_aria_model
from download_bytedance_piano_model import download_bytedance_piano_model

log("准备六声部分离资源: download_multistem_model.py")
download_multistem_model(printer=log)
log("准备 RoFormer vocal_rvc 人声分离资源: download_vocal_model.py")
download_vocal_model(printer=log)
log("准备 RoFormer karaoke 主唱/和声分离资源: download_vocal_harmony_model.py")
download_chorus_model(printer=log)
log("准备 Aria-AMT 资源: download_aria_amt_model.py")
download_aria_model(printer=log)
log("准备 ByteDance Piano 资源: download_bytedance_piano_model.py")
download_bytedance_piano_model(printer=log)

log("所有模型就绪")

In [ ]:
#@title 4️⃣ 启动 Gradio 应用（详细日志 + 公开链接）
import logging
import os
import platform
import subprocess
import sys
import tempfile
from datetime import datetime
from pathlib import Path


def tlog(message=""):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)


os.chdir("/content/music-to-midi")
try:
    subprocess.run(["fuser", "-k", "7860/tcp"], capture_output=True, text=True, timeout=5)
except Exception as exc:
    tlog(f"Skip port cleanup: {exc}")

PROJECT_ROOT = "/content/music-to-midi"
amt_src = "/content/music-to-midi/space/YourMT3/amt/src"
for p in [PROJECT_ROOT, amt_src]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_MIN_LOG_LEVEL"] = "3"
os.environ["NUMBA_CACHE_DIR"] = "/tmp/numba_cache"
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
logger = logging.getLogger("colab-app")

import gradio as gr
import torch

logger.info("========== Startup Diagnostics ==========")
logger.info(f"Python version: {platform.python_version()}")
logger.info(f"Gradio version: {gr.__version__}")
logger.info(f"Torch version: {torch.__version__}")
logger.info(f"CUDA available: {torch.cuda.is_available()}")
logger.info(f"Project root: {PROJECT_ROOT}")
logger.info(f"YourMT3 source path: {amt_src}")
logger.info(f"YourMT3 ymt3.py exists: {Path(amt_src, 'model', 'ymt3.py').exists()}")

try:
    from model.ymt3 import YourMT3  # noqa: F401
    logger.info("✅ YourMT3 code import success")
except ImportError as exc:
    raise RuntimeError(f"YourMT3 import failed: {exc}, 请重新运行第 3 步")

from src.models.data_models import Config, ProcessingStage
from src.core.pipeline import MusicToMidiPipeline
from src.utils.gpu_utils import clear_gpu_memory
from src.utils.yourmt3_downloader import DEFAULT_MODEL, OFFICIAL_YOURMT3_MODEL_KEYS, YOURMT3_MODELS

device_label = f"GPU ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "CPU"
logger.info(f"Current device: {device_label}")

model_cache_root = Path(os.path.expanduser("~/.cache/music_ai_models/yourmt3_all"))
ckpts = sorted(model_cache_root.rglob("*.ckpt")) if model_cache_root.exists() else []
logger.info(f"Model cache root: {model_cache_root}")
logger.info(f"Checkpoint count: {len(ckpts)}")

YOURMT3_MODEL_LABEL_TO_KEY = {
    YOURMT3_MODELS[model_key].get("ui_label", model_key): model_key
    for model_key in OFFICIAL_YOURMT3_MODEL_KEYS
}
YOURMT3_MODEL_CHOICES = list(YOURMT3_MODEL_LABEL_TO_KEY.keys())
DEFAULT_YOURMT3_MODEL_LABEL = next(
    label for label, model_key in YOURMT3_MODEL_LABEL_TO_KEY.items() if model_key == DEFAULT_MODEL
)
YOURMT3_PROCESSING_MODES = {"smart"}
COLAB_LANGUAGE = "zh_CN"
COLAB_I18N = {
    "zh_CN": {
        "mode.smart": "完整混音多乐器转写（SMART）",
        "mode.vocal_split": "人声/伴奏分离后分别转写（VOCAL_SPLIT）",
        "mode.six_stem_split": "六声部分离 + 分别转写",
        "mode.piano_transkun": "钢琴专用转写 (Transkun)",
        "mode.piano_aria_amt": "钢琴专用转写 (Aria-AMT)",
        "mode.piano_bytedance_pedal": "钢琴专用转写 (ByteDance Pedal)",
        "error.upload_required": "请先上传音频文件",
        "error.unknown_mode": "未知处理模式: {mode}",
        "error.unknown_yourmt3_model": "未知 YourMT3+ 模型模式: {model}",
        "error.conversion_failed": "转换失败: {error}",
        "stage.preprocessing": "预处理",
        "stage.separation": "音源分离",
        "stage.transcription": "音频转写",
        "stage.vocal_transcription": "人声转写",
        "stage.synthesis": "MIDI合成",
        "stage.complete": "完成",
        "status.complete_header": "--- 转换完成 ---",
        "status.elapsed": "耗时",
        "status.seconds": "秒",
        "status.total_notes": "总音符数",
        "status.device": "设备",
        "status.output_count": "输出文件数",
        "status.merged_midi": "合并 MIDI",
        "status.stem_midi_count": "分 stem MIDI",
        "status.stem_midi_suffix": " 个",
        "status.accompaniment_midi": "伴奏 MIDI",
        "status.vocal_midi": "人声 MIDI",
        "status.midi_file": "MIDI 文件",
        "yourmt3.model_title": "YourMT3+ 模型模式",
        "yourmt3.checkpoint": "检查点",
        "yourmt3.traits": "适合/特点",
        "output.title": "输出说明",
        "output.vocal_default": "默认输出 `_accompaniment.mid` 与 `_vocal.mid`。",
        "output.vocal_merged": "勾选合并后额外输出 `_vocal_accompaniment_merged.mid`。",
        "output.six_stem_audio": "输出分离后的六个 stem 音频。",
        "output.six_stem_midi": "固定输出六个 stem MIDI。",
        "output.six_stem_merged": "额外输出一个 all_stems 合并 MIDI。",
        "output.single_midi": "输出一个 MIDI 文件。",
        "mode_info.smart": "**SMART** — 通过所选 {model_family} 官方模型模式对完整混音进行多乐器转写，识别 {instrument_standard} 乐器并生成 MIDI。",
        "mode_info.vocal_split": "**VOCAL_SPLIT** — 使用 TelkNet 对齐的 {separator_chain} 分离人声与伴奏；音频分离本身不使用 YourMT3+，后续 MIDI 扩展固定使用工程默认 YourMT3+ noPS 后端分别转写。",
        "mode_info.six_stem_split": "**六声部分离 + 分别转写** — 使用 {separator_model} 分离 {stem_names} 六个 stem；音频分离本身不使用 YourMT3+，后续 MIDI 扩展固定使用工程默认 YourMT3+ noPS 后端生成 stem MIDI 并合并。",
        "mode_info.piano_transkun": "**{model_name}** — 钢琴专用转写，适合纯钢琴音频。",
        "mode_info.piano_aria_amt": "**{model_name}** — 钢琴专用转写，首次使用会检查 {checkpoint_name}。",
        "mode_info.piano_bytedance_pedal": "**{model_name}** — 钢琴专用带踏板转写，适合纯钢琴音频，会保留延音踏板 {pedal_cc}。",
        "ui.header": "# 🎵 Music to MIDI\n当前 Colab 版提供六种处理模式；YourMT3+ 官方模型下拉只在 SMART 完整混音模式暴露，分离模式固定使用对应分离链路，并用工程默认 YourMT3+ noPS 后端完成 MIDI 扩展。钢琴专用模式使用 Transkun、Aria-AMT 或 ByteDance Pedal。",
        "ui.audio_input": "上传音频文件",
        "ui.audio_hint": "支持 MP3, WAV, FLAC, OGG, M4A（自动转换为 WAV 处理）",
        "ui.mode_label": "处理模式",
        "ui.yourmt3_model_label": "YourMT3+ 官方模型模式",
        "ui.yourmt3_model_info": "仅 SMART 完整混音模式开放选择；分离模式使用工程默认 noPS 后端",
        "ui.vocal_merge_midi": "输出 1 个人声+伴奏合并 MIDI（人声分离模式）",
        "ui.start": "▶ 开始转换",
        "ui.device": "当前设备",
        "ui.status": "状态",
        "ui.status_placeholder": "等待转换...",
        "ui.download": "下载文件",
        "ui.logs": "控制台日志",
        "ui.footer": "<center>基于 [YourMT3+](https://github.com/mimbres/YourMT3) 官方模型模式 | [GitHub](https://github.com/mason369/music-to-midi)</center>",
        "ui.launching": "🚀 启动中... 公开链接将在下方显示（详细日志模式）",
    },
    "en_US": {
        "mode.smart": "Full-mix multi-instrument transcription (SMART)",
        "mode.vocal_split": "Vocal/accompaniment split + separate transcription (VOCAL_SPLIT)",
        "mode.six_stem_split": "Six-stem split + separate transcription",
        "mode.piano_transkun": "Piano transcription (Transkun)",
        "mode.piano_aria_amt": "Piano transcription (Aria-AMT)",
        "mode.piano_bytedance_pedal": "Piano transcription (ByteDance Pedal)",
        "error.upload_required": "Please upload an audio file first",
        "error.unknown_mode": "Unknown processing mode: {mode}",
        "error.unknown_yourmt3_model": "Unknown YourMT3+ model mode: {model}",
        "error.conversion_failed": "Conversion failed: {error}",
        "stage.preprocessing": "Preprocessing",
        "stage.separation": "Separation",
        "stage.transcription": "Transcription",
        "stage.vocal_transcription": "Vocal transcription",
        "stage.synthesis": "MIDI synthesis",
        "stage.complete": "Complete",
        "status.complete_header": "--- Conversion Complete ---",
        "status.elapsed": "Elapsed",
        "status.seconds": "seconds",
        "status.total_notes": "Total notes",
        "status.device": "Device",
        "status.output_count": "Output file count",
        "status.merged_midi": "Merged MIDI",
        "status.stem_midi_count": "Stem MIDI count",
        "status.stem_midi_suffix": "",
        "status.accompaniment_midi": "Accompaniment MIDI",
        "status.vocal_midi": "Vocal MIDI",
        "status.midi_file": "MIDI file",
        "yourmt3.model_title": "YourMT3+ model mode",
        "yourmt3.checkpoint": "Checkpoint",
        "yourmt3.traits": "Best for / traits",
        "output.title": "Output notes",
        "output.vocal_default": "Outputs `_accompaniment.mid` and `_vocal.mid` by default.",
        "output.vocal_merged": "When merge is enabled, also outputs `_vocal_accompaniment_merged.mid`.",
        "output.six_stem_audio": "Outputs the separated six stem audio files.",
        "output.six_stem_midi": "Always outputs six stem MIDI files.",
        "output.six_stem_merged": "Also outputs one all_stems merged MIDI.",
        "output.single_midi": "Outputs one MIDI file.",
        "mode_info.smart": "**SMART** — Uses the selected official {model_family} model mode to transcribe the full mix, recognize {instrument_standard} instruments, and generate MIDI.",
        "mode_info.vocal_split": "**VOCAL_SPLIT** — Uses the TelkNet-aligned {separator_chain} to separate vocals and accompaniment. Audio separation itself does not use YourMT3+; the MIDI extension uses the project-default YourMT3+ noPS backend for downstream transcription.",
        "mode_info.six_stem_split": "**Six-stem split + separate transcription** — Uses {separator_model} to separate {stem_names} stems. Audio separation itself does not use YourMT3+; the MIDI extension uses the project-default YourMT3+ noPS backend to write stem MIDI and a merged MIDI.",
        "mode_info.piano_transkun": "**{model_name}** — Piano-focused transcription for solo piano audio.",
        "mode_info.piano_aria_amt": "**{model_name}** — Piano-focused transcription; checks the {checkpoint_name} on first use.",
        "mode_info.piano_bytedance_pedal": "**{model_name}** — Pedal-aware piano transcription for solo piano audio; preserves sustain pedal {pedal_cc}.",
        "ui.header": "# 🎵 Music to MIDI\nThe Colab build provides six processing modes. The official YourMT3+ model dropdown is exposed only for SMART full-mix transcription; split modes use their fixed separation chains and the project-default YourMT3+ noPS backend for MIDI extension. Dedicated piano modes use Transkun, Aria-AMT, or ByteDance Pedal.",
        "ui.audio_input": "Upload audio file",
        "ui.audio_hint": "Supports MP3, WAV, FLAC, OGG, M4A (automatically converted to WAV for processing)",
        "ui.mode_label": "Processing mode",
        "ui.yourmt3_model_label": "Official YourMT3+ model mode",
        "ui.yourmt3_model_info": "Selectable only for SMART full-mix mode; split modes use the default noPS backend",
        "ui.vocal_merge_midi": "Output 1 merged vocal+accompaniment MIDI (vocal split mode)",
        "ui.start": "▶ Start conversion",
        "ui.device": "Current device",
        "ui.status": "Status",
        "ui.status_placeholder": "Waiting for conversion...",
        "ui.download": "Download files",
        "ui.logs": "Console logs",
        "ui.footer": "<center>Powered by official [YourMT3+](https://github.com/mimbres/YourMT3) model modes | [GitHub](https://github.com/mason369/music-to-midi)</center>",
        "ui.launching": "🚀 Launching... The public link will appear below (verbose logging mode)",
    }
}


def ct(key, **kwargs):
    text = COLAB_I18N[COLAB_LANGUAGE][key]
    return text.format(**kwargs) if kwargs else text


COLAB_MODE_IDS = (
    "smart",
    "vocal_split",
    "six_stem_split",
    "piano_transkun",
    "piano_aria_amt",
    "piano_bytedance_pedal",
)
COLAB_MODE_CHOICES = [(ct(f"mode.{mode_id}"), mode_id) for mode_id in COLAB_MODE_IDS]
COLAB_MODE_INFO_TERMS = {
    "smart": {"model_family": "YourMT3+", "instrument_standard": "GM"},
    "vocal_split": {"separator_chain": "RoFormer vocal_rvc/karaoke ensemble"},
    "six_stem_split": {"separator_model": "BS-RoFormer SW", "stem_names": "bass/drums/guitar/piano/vocals/other"},
    "piano_transkun": {"model_name": "Transkun"},
    "piano_aria_amt": {"model_name": "Aria-AMT", "checkpoint_name": "Aria-AMT checkpoint"},
    "piano_bytedance_pedal": {"model_name": "ByteDance Pedal", "pedal_cc": "CC64"},
}


def uses_yourmt3_processing_mode(mode):
    return mode in YOURMT3_PROCESSING_MODES


LOG_FILE = "/tmp/midi_process.log"


class _RobustFileHandler(logging.Handler):
    def __init__(self, filename):
        super().__init__()
        self.filename = filename

    def emit(self, record):
        try:
            msg = self.format(record)
            with open(self.filename, "a", encoding="utf-8") as f:
                f.write(msg + "\n")
        except Exception:
            pass


_fh = _RobustFileHandler(LOG_FILE)
_fh.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S"))
_fh.setLevel(logging.INFO)
for _name in ("colab-app", "src.core", "src.utils"):
    logging.getLogger(_name).addHandler(_fh)


def clear_logs():
    try:
        open(LOG_FILE, "w", encoding="utf-8").close()
    except Exception:
        pass


def read_logs():
    try:
        with open(LOG_FILE, "r", encoding="utf-8", errors="replace") as f:
            content = f.read().replace("\x00", "")
        lines = content.strip().split("\n")
        return "\n".join(lines[-200:]) if lines and lines[0] else ""
    except Exception as exc:
        return f"[error] {exc}"


def ensure_aria_amt_weights():
    from download_aria_amt_model import download_aria_model, is_aria_model_available

    if is_aria_model_available():
        logger.info("Aria-AMT checkpoint found")
        return
    logger.info("Aria-AMT checkpoint not found, downloading via download_aria_amt_model.py")
    download_aria_model(printer=logger.info)


def convert_audio_to_midi(
    audio_path,
    mode,
    yourmt3_model_label,
    vocal_split_merge_midi=False,
    progress=gr.Progress(),
):
    if audio_path is None:
        raise gr.Error(ct("error.upload_required"))

    clear_logs()
    logger.info("========== New Conversion ==========")
    logger.info(f"Audio file: {Path(audio_path).name}")
    logger.info(f"Mode: {mode}")

    if mode not in COLAB_MODE_IDS:
        raise gr.Error(ct("error.unknown_mode", mode=mode))

    config = Config()
    config.processing_mode = mode
    yourmt3_model_key = YOURMT3_MODEL_LABEL_TO_KEY.get(yourmt3_model_label)
    if yourmt3_model_key is None:
        raise gr.Error(ct("error.unknown_yourmt3_model", model=yourmt3_model_label))
    if config.processing_mode in YOURMT3_PROCESSING_MODES:
        config.yourmt3_model = yourmt3_model_key
        logger.info(f"YourMT3 model mode: {yourmt3_model_label} ({yourmt3_model_key})")
    config.vocal_split_merge_midi = bool(config.processing_mode == "vocal_split" and vocal_split_merge_midi)

    if config.processing_mode == "piano_aria_amt":
        ensure_aria_amt_weights()
    elif config.processing_mode == "piano_bytedance_pedal":
        from download_bytedance_piano_model import download_bytedance_piano_model
        download_bytedance_piano_model(printer=logger.info)

    pipeline = MusicToMidiPipeline(config)
    output_dir = tempfile.mkdtemp(prefix="midi_output_")
    logger.info(f"Output dir: {output_dir}")

    def on_progress(p):
        stage_name = {
            ProcessingStage.PREPROCESSING: ct("stage.preprocessing"),
            ProcessingStage.SEPARATION: ct("stage.separation"),
            ProcessingStage.TRANSCRIPTION: ct("stage.transcription"),
            ProcessingStage.VOCAL_TRANSCRIPTION: ct("stage.vocal_transcription"),
            ProcessingStage.SYNTHESIS: ct("stage.synthesis"),
            ProcessingStage.COMPLETE: ct("stage.complete"),
        }.get(p.stage, str(p.stage))
        logger.info(f"Progress {p.overall_progress * 100:5.1f}% [{stage_name}] {p.message}")
        progress(p.overall_progress, desc=f"[{stage_name}] {p.message}")

    try:
        result = pipeline.process(audio_path=audio_path, output_dir=output_dir, progress_callback=on_progress)
    except Exception as exc:
        logger.error(f"Conversion failed: {exc}")
        raise gr.Error(ct("error.conversion_failed", error=exc)) from exc
    finally:
        try:
            clear_gpu_memory()
            logger.info("GPU memory cleaned")
        except Exception as exc:
            logger.warning(f"GPU memory cleanup warning: {exc}")

    output_files = []
    if result.midi_path and Path(result.midi_path).exists():
        output_files.append(result.midi_path)
    if result.stem_midi_paths:
        for stem_midi_path in result.stem_midi_paths.values():
            if stem_midi_path and Path(stem_midi_path).exists():
                output_files.append(stem_midi_path)
    if result.vocal_midi_path and Path(result.vocal_midi_path).exists():
        output_files.append(result.vocal_midi_path)
    if result.accompaniment_midi_path and Path(result.accompaniment_midi_path).exists():
        if result.accompaniment_midi_path != result.midi_path:
            output_files.append(result.accompaniment_midi_path)
    if result.separated_audio:
        for audio_file in result.separated_audio.values():
            if audio_file and Path(audio_file).exists():
                output_files.append(audio_file)

    output_files = list(dict.fromkeys(output_files))
    logger.info(f"Output file count: {len(output_files)}")

    bpm_str = f"{result.beat_info.bpm:.1f}" if result.beat_info else "N/A"
    status_lines = [
        ct("status.complete_header"),
        f"{ct('status.elapsed')}: {result.processing_time:.1f} {ct('status.seconds')}",
        f"{ct('status.total_notes')}: {result.total_notes}",
        f"BPM: {bpm_str}",
        f"{ct('status.device')}: {device_label}",
        f"{ct('status.output_count')}: {len(output_files)}",
    ]
    if result.stem_midi_paths:
        status_lines.append(f"{ct('status.merged_midi')}: {Path(result.midi_path).name}")
        status_lines.append(f"{ct('status.stem_midi_count')}: {len(result.stem_midi_paths)}{ct('status.stem_midi_suffix')}")
    elif result.vocal_midi_path:
        status_lines.append(f"{ct('status.accompaniment_midi')}: {Path(result.accompaniment_midi_path).name}")
        status_lines.append(f"{ct('status.vocal_midi')}: {Path(result.vocal_midi_path).name}")
        if result.merged_midi_path:
            status_lines.append(f"{ct('status.merged_midi')}: {Path(result.merged_midi_path).name}")
    else:
        status_lines.append(f"{ct('status.midi_file')}: {Path(result.midi_path).name}")

    logger.info("========== Conversion Finished ==========")
    return output_files, "\n".join(status_lines)


def update_mode_info(mode):
    if mode not in COLAB_MODE_IDS:
        raise gr.Error(ct("error.unknown_mode", mode=mode))
    return ct(f"mode_info.{mode}", **COLAB_MODE_INFO_TERMS[mode])


def update_yourmt3_model_info(yourmt3_model_label):
    model_key = YOURMT3_MODEL_LABEL_TO_KEY.get(yourmt3_model_label, DEFAULT_MODEL)
    model_info = YOURMT3_MODELS[model_key]
    label = model_info.get("ui_label", yourmt3_model_label)
    is_zh = COLAB_LANGUAGE.startswith("zh")
    if is_zh:
        description = model_info.get("description") or model_info.get("ui_description", "")
        feature_items = model_info.get("features_zh") or model_info.get("features", [])
        separator = "："
        features = "，".join(feature_items)
    else:
        description = model_info.get("ui_description") or model_info.get("description", "")
        feature_items = model_info.get("features_en") or model_info.get("features", [])
        separator = ": "
        features = ", ".join(feature_items)
    checkpoint = model_info.get("checkpoint", "")
    return (
        f"**{ct('yourmt3.model_title')}{separator}{label}**\n\n"
        f"{description}\n\n"
        f"**{ct('yourmt3.checkpoint')}**{separator}`{checkpoint}`\n\n"
        f"**{ct('yourmt3.traits')}**{separator}{features}"
    )


def update_output_info(mode):
    if mode == "vocal_split":
        return f"**{ct('output.title')}**\n\n- {ct('output.vocal_default')}\n- {ct('output.vocal_merged')}"
    if mode == "six_stem_split":
        return f"**{ct('output.title')}**\n\n- {ct('output.six_stem_audio')}\n- {ct('output.six_stem_midi')}\n- {ct('output.six_stem_merged')}"
    return f"**{ct('output.title')}**\n\n- {ct('output.single_midi')}"


def update_mode_controls(mode):
    is_vocal = mode == "vocal_split"
    uses_yourmt3 = uses_yourmt3_processing_mode(mode)
    return (
        update_mode_info(mode),
        update_output_info(mode),
        gr.update(visible=uses_yourmt3),
        gr.update(visible=uses_yourmt3),
        gr.update(visible=is_vocal),
    )


LOG_POLL_JS = """<script>
(function() {
    var _pollTimer = setInterval(function() {
        var ta = document.querySelector('.log-box textarea');
        if (!ta) return;
        var setter = Object.getOwnPropertyDescriptor(HTMLTextAreaElement.prototype, 'value').set;
        fetch('./api/read_logs', {method: 'POST', headers: {'Content-Type': 'application/json'}, body: JSON.stringify({data: []})})
        .then(function(r) { return r.json(); })
        .then(function(json) {
            var logText = (json.data && json.data[0]) ? json.data[0] : '';
            if (logText) setter.call(ta, logText);
            ta.dispatchEvent(new Event('input', {bubbles: true}));
            ta.scrollTop = ta.scrollHeight;
        })
        .catch(function() {});
    }, 1200);
})();
</script>"""

with gr.Blocks(
    title="Music to MIDI (Colab GPU)",
    head=LOG_POLL_JS,
    theme=gr.themes.Base(primary_hue=gr.themes.colors.blue, neutral_hue=gr.themes.colors.slate),
) as demo:
    gr.Markdown(ct("ui.header"))

    with gr.Row():
        with gr.Column(scale=5):
            audio_input = gr.Audio(label=ct("ui.audio_input"), type="filepath")
            gr.Markdown(f"<small>{ct('ui.audio_hint')}</small>")
            mode_radio = gr.Radio(
                COLAB_MODE_CHOICES,
                value="smart",
                label=ct("ui.mode_label"),
            )
            mode_info = gr.Markdown(update_mode_info("smart"))
            yourmt3_model_dropdown = gr.Dropdown(
                choices=YOURMT3_MODEL_CHOICES,
                value=DEFAULT_YOURMT3_MODEL_LABEL,
                label=ct("ui.yourmt3_model_label"),
                info=ct("ui.yourmt3_model_info"),
            )
            yourmt3_model_info = gr.Markdown(update_yourmt3_model_info(DEFAULT_YOURMT3_MODEL_LABEL))
            output_info = gr.Markdown(update_output_info("smart"))
            vocal_split_merge_midi = gr.Checkbox(value=False, label=ct("ui.vocal_merge_midi"), visible=False)
            mode_radio.change(
                fn=update_mode_controls,
                inputs=[mode_radio],
                outputs=[mode_info, output_info, yourmt3_model_dropdown, yourmt3_model_info, vocal_split_merge_midi],
                api_name=False,
            )
            yourmt3_model_dropdown.change(
                fn=update_yourmt3_model_info,
                inputs=[yourmt3_model_dropdown],
                outputs=[yourmt3_model_info],
                api_name=False,
            )
            convert_btn = gr.Button(ct("ui.start"), variant="primary", size="lg")
            gr.Markdown(f"{ct('ui.device')}: **{device_label}**")

        with gr.Column(scale=5):
            status_output = gr.Textbox(label=ct("ui.status"), interactive=False, lines=7, placeholder=ct("ui.status_placeholder"))
            file_output = gr.File(label=ct("ui.download"), file_count="multiple")
            log_output = gr.Textbox(label=ct("ui.logs"), interactive=False, lines=16, max_lines=30, elem_classes="log-box")

    convert_btn.click(
        fn=convert_audio_to_midi,
        inputs=[audio_input, mode_radio, yourmt3_model_dropdown, vocal_split_merge_midi],
        outputs=[file_output, status_output],
    )

    _log_poll_btn = gr.Button(visible=False)
    _log_poll_btn.click(fn=read_logs, inputs=[], outputs=[log_output], api_name="read_logs", queue=False)

    gr.Markdown(ct("ui.footer"))

print("\n" + "=" * 60)
print(ct("ui.launching"))
print("=" * 60 + "\n")
demo.launch(share=True, server_name="0.0.0.0", server_port=7860)
